# PYNQ 2.5 — ECG Waveform Capture

Tests the ECG demo bitstream on PYNQ 2.5 (no FastAPI / uvicorn required).
Loads the overlay, writes startup config, captures 5 seconds of ECG samples,
and plots DAC / RAW / FILTERED traces inline.

### Prerequisites
- `ecg_demo.bit` and `ecg_demo.hwh` are in the **same folder** as this notebook  
  (`/home/xilinx/pynq-ecg-demo/ps/`)
- PMOD DA4 plugged into **JA** (SPI DAC)
- PMOD AD2 plugged into **right half of JB** (pins JB3/JB4 = SCL/SDA)
- Loopback wire: DAC Ch A output → ADC Ch 0 input

In [ ]:
# Cell 1 — Load bitstream
# Tries Overlay (preferred); falls back to raw MMIO if HWH parsing fails on PYNQ 2.5

import os
import time
import numpy as np

BITSTREAM = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'ecg_demo.bit')
AXI_BASE  = 0x43C00000
AXI_SIZE  = 256   # bytes (covers all 17 registers)

try:
    from pynq import Overlay
    ol = Overlay(BITSTREAM)
    # axi_ecg_ctrl is the IP core name set in the Vivado block design
    mmio = ol.axi_ecg_ctrl
    print(f'Overlay loaded: {BITSTREAM}')
    print('IP cores found:', list(ol.ip_dict.keys()))
    use_overlay = True
except Exception as e:
    print(f'Overlay failed ({e})')
    print('Falling back to raw MMIO — bitstream must be pre-loaded or use xlnk.')
    from pynq import MMIO
    mmio = MMIO(AXI_BASE, AXI_SIZE)
    use_overlay = False
    print(f'MMIO at 0x{AXI_BASE:08X}, {AXI_SIZE} bytes')

print('\nReady.')

In [ ]:
# Cell 2 — Write startup config and verify readback
# Register map from handoffs/register_map.md

# Offsets
REG_BPM_CH_A   = 0x00   # R/W  Heart rate channel A (default 60 BPM)
REG_RR_FLUCT   = 0x04   # R/W  RR interval jitter   (0 = none)
REG_AMP_FLUCT  = 0x08   # R/W  Amplitude jitter     (0 = none)
REG_THRESHOLD  = 0x38   # R/W  R-peak detect threshold (2983 from algorithm_spec.md)
REG_ECG_RAW    = 0x28   # R    Raw ADC sample (12-bit)
REG_ECG_FILT   = 0x2C   # R    FIR-filtered sample (12-bit)
REG_BPM_OUT    = 0x30   # R    Detected BPM
REG_RPEAK_CNT  = 0x34   # R    R-peak counter (wraps at 65535)
REG_STATUS     = 0x3C   # R    Bit0=signal_present, Bit1=lead_off
REG_ECG_DAC    = 0x40   # R    DAC output monitor (12-bit)

STARTUP_CONFIG = [
    (REG_BPM_CH_A,  60),    # 60 BPM
    (REG_RR_FLUCT,   0),    # no RR jitter
    (REG_AMP_FLUCT,  0),    # no amplitude variation
    (REG_THRESHOLD, 2983),  # threshold from algorithm_spec.md
]

print('Writing startup config...')
for offset, value in STARTUP_CONFIG:
    mmio.write(offset, value)

print(f'  BPM_CH_A   = {mmio.read(REG_BPM_CH_A)}')
print(f'  RR_FLUCT   = {mmio.read(REG_RR_FLUCT)}')
print(f'  AMP_FLUCT  = {mmio.read(REG_AMP_FLUCT)}')
print(f'  THRESHOLD  = {mmio.read(REG_THRESHOLD)}')
print('Config written OK.')

In [ ]:
# Cell 3 — Capture 5 seconds of ECG samples (1800 samples @ 360 Hz)

FS       = 360
DURATION = 5      # seconds
N        = FS * DURATION

raw_samples      = np.zeros(N, dtype=np.int32)
filtered_samples = np.zeros(N, dtype=np.int32)
dac_samples      = np.zeros(N, dtype=np.int32)
rpeak_count      = np.zeros(N, dtype=np.int32)
bpm_samples      = np.zeros(N, dtype=np.int32)

print(f'Capturing {N} samples ({DURATION} s) ...')
interval = 1.0 / FS
t_start  = time.monotonic()

for i in range(N):
    t_sample = time.monotonic()
    raw_samples[i]      = mmio.read(REG_ECG_RAW)  & 0xFFF
    filtered_samples[i] = mmio.read(REG_ECG_FILT) & 0xFFF
    dac_samples[i]      = mmio.read(REG_ECG_DAC)  & 0xFFF
    rpeak_count[i]      = mmio.read(REG_RPEAK_CNT) & 0xFFFF
    bpm_samples[i]      = mmio.read(REG_BPM_OUT)  & 0xFF

    if (i + 1) % (FS // 2) == 0:
        print(f'  {i+1:4d}/{N}  RAW={raw_samples[i]:4d}  '
              f'FILT={filtered_samples[i]:4d}  '
              f'BPM={bpm_samples[i]:3d}')

    elapsed = time.monotonic() - t_sample
    sleep_t = interval - elapsed
    if sleep_t > 0:
        time.sleep(sleep_t)

total_time = time.monotonic() - t_start
print(f'\nDone. Captured in {total_time:.2f} s  '
      f'(effective rate: {N/total_time:.1f} Hz)')
print(f'RAW   min={raw_samples.min():4d}  max={raw_samples.max():4d}')
print(f'FILT  min={filtered_samples.min():4d}  max={filtered_samples.max():4d}')
print(f'DAC   min={dac_samples.min():4d}  max={dac_samples.max():4d}')

In [ ]:
# Cell 4 — Plot ECG waveform (inline matplotlib)

import matplotlib
matplotlib.use('Agg')   # works headless in JupyterLab
%matplotlib inline
import matplotlib.pyplot as plt

t_axis = np.arange(N) / FS  # time in seconds

# Detect R-peak edges from rpeak_count increments
rpeak_edges = np.where(np.diff(rpeak_count.astype(np.int32)) > 0)[0]

# Mean BPM over capture
bpm_nonzero = bpm_samples[bpm_samples > 0]
mean_bpm = int(np.mean(bpm_nonzero)) if len(bpm_nonzero) else 0

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.patch.set_facecolor('#0E1117')

plot_cfg = [
    (axes[0], dac_samples,      '#4488FF', 'DAC output (12-bit)'),
    (axes[1], raw_samples,      '#AAAAAA', 'ADC raw   (12-bit)'),
    (axes[2], filtered_samples, '#00FF88', 'FIR filtered (12-bit)'),
]

for ax, data, color, label in plot_cfg:
    ax.set_facecolor('#0E1117')
    ax.plot(t_axis, data, color=color, linewidth=0.8, label=label)
    ax.set_ylabel('ADC counts', color='white', fontsize=9)
    ax.tick_params(colors='white', labelsize=8)
    ax.spines[:].set_color('#444444')
    ax.legend(loc='upper right', fontsize=8,
               facecolor='#1a1a2e', labelcolor='white')
    ax.set_ylim(0, 4095)
    for edge in rpeak_edges:
        ax.axvline(x=t_axis[edge], color='#FF4444', alpha=0.5,
                   linewidth=0.8, linestyle='--')

axes[-1].set_xlabel('Time (s)', color='white', fontsize=9)
fig.suptitle(
    f'PYNQ-Z2 ECG Capture — {DURATION} s  |  Mean BPM: {mean_bpm}  '
    f'|  R-peaks: {len(rpeak_edges)}',
    color='white', fontsize=11,
)
plt.tight_layout()
plt.savefig('ecg_capture.png', dpi=120, bbox_inches='tight',
            facecolor='#0E1117')
plt.show()
print('Plot saved to ecg_capture.png')

In [ ]:
# Cell 5 — STATUS register check

status_raw     = mmio.read(REG_STATUS) & 0x03
signal_present = bool(status_raw & 0x01)
lead_off       = bool(status_raw & 0x02)

print(f'STATUS register : 0x{status_raw:02X}')
print(f'  signal_present : {signal_present}')
print(f'  lead_off       : {lead_off}')
print()

if not signal_present:
    print('WARNING: signal_present is NOT set.')
    print('  Check that:')
    print('  1. PMOD DA4 is plugged into JA')
    print('  2. PMOD AD2 is on the RIGHT HALF of JB (pins JB3/JB4)')
    print('  3. Loopback wire connects DAC Ch A output to ADC Ch 0 input')
    print('  4. ECG_RAW samples above are non-zero')
    raw_nonzero = np.sum(raw_samples > 16)
    print(f'  RAW samples > 16: {raw_nonzero}/{N}')
else:
    print('PASS: signal_present is asserted — ECG signal detected.')
    if lead_off:
        print('WARNING: lead_off is set — check ADC connection / loopback wire.')
    else:
        print('PASS: lead_off is clear — loopback connection OK.')

print()
print('All cells complete. Board is ready for ps/server.py after upgrading to PYNQ 3.0.')